# Data Check: Comparing c_data (Chengyang) vs t_data (Tahreem)
Merge both annotation datasets on `vaers_id` using a full outer join, then investigate discrepancies in `llm_result_final`.

In [ ]:
import pandas as pd
import numpy as np

c_data = pd.read_csv('data/o4mini_annotation_final_chengyang_version.csv')
t_data = pd.read_csv('data/annotated_data.csv')

print(f'c_data shape: {c_data.shape}')
print(f't_data shape: {t_data.shape}')

## Full Outer Join
Merge on `vaers_id`, suffixing c_data columns with `_ch` and t_data columns with `_ta`.

In [ ]:
merged = c_data.merge(t_data, on='vaers_id', how='outer', suffixes=('_ch', '_ta'))
print(f'Merged shape: {merged.shape}')

only_ch = merged[merged['final_result_ta'].isna()]
only_ta = merged[merged['final_result_ch'].isna()]
print(f'Records only in c_data: {len(only_ch)}')
print(f'Records only in t_data: {len(only_ta)}')
print(f'Records in both:        {len(merged) - len(only_ch) - len(only_ta)}')

## Select Relevant Columns
Pick `vaers_id`, `final_result`, and `llm_result_final` from both datasets.

In [ ]:
cols = ['vaers_id', 'final_result_ch', 'final_result_ta', 'llm_result_final_ch', 'llm_result_final_ta']
df = merged[cols].copy()
df.head()

## Filter Records Where `llm_result_final` Differs
Use `fillna` with a sentinel so NaN vs NaN is correctly treated as equal (not flagged as a difference).

In [ ]:
SENTINEL = '__NO_TEMPORAL__'

diff_mask = (
    df['llm_result_final_ch'].fillna(SENTINEL)
    != df['llm_result_final_ta'].fillna(SENTINEL)
)

diff_df = df[diff_mask].copy()
print(f'Records with different llm_result_final: {len(diff_df)}')
diff_df

## `llm_result_final` Distribution
Understand how `llm_result_final` is distributed in each dataset.

In [ ]:
def summarize_llm_result(series, label):
    no_temp = (series == 'No temporal information found').sum()
    nan_count = series.isna().sum()
    has_temporal = len(series) - no_temp - nan_count
    print(f'{label}:')
    print(f'  Has temporal grouping : {has_temporal}')
    print(f'  No temporal info found: {no_temp}')
    print(f'  NaN (final_result==2)  : {nan_count}')
    print()

summarize_llm_result(df['llm_result_final_ch'], 'c_data (Chengyang)')
summarize_llm_result(df['llm_result_final_ta'], 't_data (Tahreem)')

## Records Where One Dataset Has "No Temporal Information Found" and the Other Doesn't

In [ ]:
NO_TEMP = 'No temporal information found'

# c_data says No temporal info, t_data has a different value
ch_no_temp_ta_diff = df[
    (df['llm_result_final_ch'] == NO_TEMP) &
    (df['llm_result_final_ta'].fillna(SENTINEL) != NO_TEMP)
].copy()
print(f'c_data has No temporal info but t_data differs: {len(ch_no_temp_ta_diff)}')
ch_no_temp_ta_diff[['vaers_id', 'final_result_ch', 'final_result_ta', 'llm_result_final_ta']]

In [ ]:
# t_data says No temporal info, c_data has a different value
ta_no_temp_ch_diff = df[
    (df['llm_result_final_ta'] == NO_TEMP) &
    (df['llm_result_final_ch'].fillna(SENTINEL) != NO_TEMP)
].copy()
print(f't_data has No temporal info but c_data differs: {len(ta_no_temp_ch_diff)}')
ta_no_temp_ch_diff[['vaers_id', 'final_result_ch', 'final_result_ta', 'llm_result_final_ch']]

## Identify the Problematic Records in c_data
Records where `llm_result_final == 'No temporal information found'` AND `final_result == 0`  
(LLM said no temporal info, annotator marked **incorrect**, but no corrected version was provided — incomplete annotations).

In [ ]:
c_data_no_temp_incorrect = c_data[
    (c_data['llm_result_final'] == NO_TEMP) &
    (c_data['final_result'] == 0)
].copy()

print(f'c_data — LLM: No temporal info AND annotator: INCORRECT (final_result==0): {len(c_data_no_temp_incorrect)}')
c_data_no_temp_incorrect[['vaers_id', 'final_result', 'llm_result_final', 'final_corrected_version', 'final_notes']]

In [ ]:
t_data_no_temp_incorrect = t_data[
    (t_data['llm_result_final'] == NO_TEMP) &
    (t_data['final_result'] == 0)
].copy()

print(f't_data — LLM: No temporal info AND annotator: INCORRECT (final_result==0): {len(t_data_no_temp_incorrect)}')
t_data_no_temp_incorrect[['vaers_id', 'final_result', 'llm_result_final', 'final_corrected_version', 'final_notes']]

In [ ]:
# View symptom_text for problematic c_data records to understand why annotator disagreed
if len(c_data_no_temp_incorrect) > 0:
    pd.set_option('display.max_colwidth', 300)
    display(c_data_no_temp_incorrect[['vaers_id', 'symptom_text', 'final_result', 'final_notes', 'final_corrected_version']])

### Breakdown of the Problematic Records
Among the records marked as incorrect (`final_result == 0`) with `llm_result_final == 'No temporal information found'`:
- **Most records** have `final_notes == 'no temporal information found'` → annotator likely agreed with LLM but mislabeled `final_result` as 0 instead of 1.
- **A small subset** have substantive notes (missing symptoms, formatting issues, etc.) → these are truly incomplete annotations where the corrected version was never provided.

In [ ]:
# Split the 227 problematic records by note type
mislabeled = c_data_no_temp_incorrect[
    c_data_no_temp_incorrect['final_notes'].str.lower().fillna('') == 'no temporal information found'
]
truly_problematic = c_data_no_temp_incorrect[
    c_data_no_temp_incorrect['final_notes'].str.lower().fillna('') != 'no temporal information found'
]

print(f'Likely mislabeled (note says no temporal info, but final_result==0): {len(mislabeled)}')
print(f'Truly problematic (other notes, correction missing)              : {len(truly_problematic)}')

### The Truly Problematic Records
Records where the annotator flagged a real issue but never provided a corrected temporal grouping.

In [ ]:
pd.set_option('display.max_colwidth', 300)
display(truly_problematic[['vaers_id', 'symptom_text', 'final_result', 'final_notes', 'final_corrected_version']])

---
## `llm_result_final` vs `final_corrected_version`: Mismatch Analysis

**Key insight:** `llm_result_final` is *derived* from `final_corrected_version` via this logic:
- `final_result == 2` → `llm_result_final = NaN` (ignores `final_corrected_version`)
- `final_result in [0,1]` + `final_corrected_version` is NaN → uses `llm_result_formatted`
- `final_result in [0,1]` + `final_corrected_version` is set → uses `final_corrected_version`

So `llm_result_final` with temporal grouping but `final_corrected_version == "No temporal info"` **cannot exist**. There are two related patterns worth examining instead.

### Pattern 1: `final_corrected_version == "No temporal info"` but `llm_result_final` is NaN
These are records where `final_result == 2` (uncertain) — the function returns NaN and ignores the corrected version entirely.

In [ ]:
NO_TEMP = 'No temporal information found'

# final_corrected_version says No temporal, but llm_result_final is NaN (final_result == 2)
pattern1 = c_data[
    (c_data['final_corrected_version'] == NO_TEMP) &
    (c_data['llm_result_final'].isna())
]
print(f'Pattern 1 — final_corrected_version=NO_TEMP but llm_result_final=NaN (final_result==2): {len(pattern1)}')
pd.set_option('display.max_colwidth', 120)
pattern1[['vaers_id', 'final_result', 'llm_result_formatted', 'final_corrected_version', 'llm_result_final', 'final_notes']]

### Pattern 2 (Reverse): `llm_result_formatted == "No temporal info"` but annotator corrected to actual grouping
These 10 records are where the LLM missed temporal structure entirely, and the annotator provided the correct grouping in `final_corrected_version`. The `final_notes` column says "temporal information found" for all of them.

In [ ]:
# LLM said No temporal info, but annotator corrected to an actual temporal grouping
pattern2 = c_data[
    (c_data['llm_result_formatted'] == NO_TEMP) &
    (c_data['llm_result_final'] != NO_TEMP) &
    (c_data['llm_result_final'].notna())
]
print(f'Pattern 2 — LLM said No temporal, annotator corrected to grouping: {len(pattern2)}')
pd.set_option('display.max_colwidth', 120)
pattern2[['vaers_id', 'final_result', 'llm_result_formatted', 'final_corrected_version', 'llm_result_final', 'final_notes']]

In [ ]:
# View the symptom_text for pattern 2 to see why LLM missed the temporal structure
pd.set_option('display.max_colwidth', 400)
display(pattern2[['vaers_id', 'symptom_text', 'llm_result_formatted', 'llm_result_final', 'final_notes']])